# Neural Prototyping

# Google Colab Mounting

In [ ]:
# !git clone https://github.com/BillyBrothers/credit-risk-modeling.git

# import sys 
# sys.path.append('/content/credit-risk-modeling/src')

Cloning into 'credit-risk-modeling'...
remote: Enumerating objects: 1025, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 1025 (delta 109), reused 117 (delta 55), pack-reused 852 (from 1)
Receiving objects: 100% (1025/1025), 53.25 MiB | 16.29 MiB/s, done.
Resolving deltas: 100% (683/683), done.
Updating files: 100% (67/67), done.


In [33]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
# from pyampute.exploration.md_patterns import mdPatterns
# from pyampute.exploration.mcar_statistical_tests import MCARTest
# import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
# import statsmodels.api as sm
# import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
# from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay
#from credit_risk_modeling import model_eval

import tensorflow as tf
from tensorflow import keras
from keras import layers

In [42]:
!ls /content/credit-risk-modeling/data/interim

credit_risk_dataset_prepped.csv  y_test.csv  y_train.csv  y_val.csv


# Imports

In [43]:
X_train = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_train_neural.csv"
)

In [44]:
X_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_test_neural.csv"
)

In [45]:
X_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_val_neural.csv"
)

In [46]:
y_train= pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_train.csv"
)
y_train = y_train.values.ravel()
neg, pos = np.bincount(y_train)
total = neg + pos
print(f"Examples:\n The training sets total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The training sets total amount of samples: 22686
 Positive: 4962 (21.87% of total)


In [47]:
y_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_test.csv"
)
y_test = y_test.values.ravel()
neg, pos = np.bincount(y_test)
total = neg + pos
print(f"Examples:\n The testing set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The testing set total amount of samples: 2917
 Positive: 638 (21.87% of total)


In [48]:
y_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_val.csv"
)
y_val = y_val.values.ravel()
neg, pos = np.bincount(y_val)
total = neg + pos
print(f"Examples:\n The validation set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The validation set total amount of samples: 6806
 Positive: 1488 (21.86% of total)


# Build Sequential Model

### 1 layered MLP Classifier

In [ ]:
model = keras.Sequential(name='MLP_1_layer')
model.add(keras.Input(shape=(X_train.shape[1], ))),
model.add(layers.Dense(units=64, activation='relu')),
model.add(layers.Dropout(rate= 0.20)),
model.summary(),
model.add(layers.Dense(units=1,activation='sigmoid')),
model.summary()

In [78]:
model.compile(
    optimizer='adam',
    loss= keras.losses.BinaryCrossentropy(),
    metrics= [keras.metrics.AUC()]
)

In [82]:
reduce_lr_plateau = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    verbose=1,
    min_lr=0.001
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=1e-2,
    patience=10,
    verbose=1
)

In [ ]:
log_dir = "logs/fit/"
tensorboard = keras.callbacks.TensorBoard(
    log_dir= log_dir
)

In [ ]:
class_weight = {
    0: 1.0,
    1: 2.0
    }

In [ ]:
model.fit(
    x= X_train,
    y= y_train,
    batch_size=32,
    epochs= 100,
    verbose=2,
    callbacks= [early_stopping, reduce_lr_plateau, tensorboard]
    validation_data= (X_val, y_val),
    class_weight= class_weight
)